# 一,本地部署推理

### 1.请求的预处理

In [28]:
from transformers import AutoTokenizer

# 初始化 Tokenizer（minimind-3 自带 chat_template）
tokenizer = AutoTokenizer.from_pretrained("/Users/wjx/Documents/wjx/minimind/minimind-3")
# 示例输入
system_message = {"role": "system", "content": "你是一个历史老师，当学生询问你问题的时候，你总是中文进行回答。"}
user_input = "请问中国的四大发明是什么？"
message = [
    system_message,
    {"role": "user", "content": user_input}
]
# 分词+编码
input_ids = tokenizer.apply_chat_template(message, add_generation_prompt=True, return_tensors="pt")
# 转换消息为模型输入格式
print(f'输入：{user_input}\n编码：{input_ids}')

输入：请问中国的四大发明是什么？
编码：tensor([[   1,  118, 4849,  234,  441, 1001, 1874, 2152, 2025,  294,  899, 2278,
         6090,  441, 5523, 4524,  294,  441, 6056, 3606,  815, 3854,  302,    2,
          234,    1,  832,  311,  234, 5736, 1405,  296, 2405,  565,  532,  851,
         3013,  776,    2,  234,    1, 1388,  570,  811,  234,   25,  234,  234,
           26,  234,  234]])


### 2.推理过程

In [29]:
from transformers import AutoModelForCausalLM
import torch

# 加载模型
model_id = "/Users/wjx/Documents/wjx/minimind/minimind-3"
model = AutoModelForCausalLM.from_pretrained(
    model_id, dtype=torch.bfloat16,  # 半精度推理
    device_map="auto"
)
input_ids = input_ids.to(model.device)
# 终止符：minimind-3 的 eos 是 <|im_end|>（id=2）；<|eot_id|> 是 Llama 的标记，此词表中不存在
terminators = [tokenizer.eos_token_id]
outputs = model.generate(input_ids,
                         max_new_tokens=256,
                         eos_token_id=terminators,
                         do_sample=True, temperature=0.6,
                         top_p=0.9)
print(f"推理结果：{outputs}")

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


推理结果：tensor([[   1,  118, 4849,  234,  441, 1001, 1874, 2152, 2025,  294,  899, 2278,
         6090,  441, 5523, 4524,  294,  441, 6056, 3606,  815, 3854,  302,    2,
          234,    1,  832,  311,  234, 5736, 1405,  296, 2405,  565,  532,  851,
         3013,  776,    2,  234,    1, 1388,  570,  811,  234,   25,  234,  234,
           26,  234,  234, 1405,  296, 2405,  565,  532,  851,  357, 1394, 3802,
          617,  345, 2357,  318,  151,  617,  345, 2329, 2381,  334, 1117, 2604,
         3142,  294, 1455,  369, 1460,  579,  296, 1209,  764, 6102,  423,  963,
         2252,  294,  461, 1950, 1874, 2636, 5686, 5086, 2214,  302,    2]],
       device='mps:0')


### 3.后处理与返回结果

In [30]:
import json
import xml.etree.ElementTree as ET

# 解码和过滤
output_text = tokenizer.decode(outputs[0][input_ids.shape[-1]:])
print(f"问题:{user_input}\n回复：{output_text}")

# json格式化
json_output = {"问题": user_input, "回复": output_text}
json_formatted = json.dumps(json_output, ensure_ascii=False, indent=4)
print("JSON格式化输出：")
print(json_formatted)

# xml格式化
root = ET.Element("Response")
question = ET.SubElement(root, "问题")
question.text = user_input
answer = ET.SubElement(root, "回复")
answer.text = output_text
xml_formatted = ET.tostring(root, encoding="utf-8").decode("utf-8")
print("XML格式化输出：")
print(xml_formatted)



问题:请问中国的四大发明是什么？
回复：中国的四大发明是造纸术、印刷术、火药和指南针，它们在各自的领域内发挥了重要作用，对世界历史的发展产生了深远的影响
JSON格式化输出：
{
    "问题": "请问中国的四大发明是什么？",
    "回复": "中国的四大发明是造纸术、印刷术、火药和指南针，它们在各自的领域内发挥了重要作用，对世界历史的发展产生了深远的影响"
}
XML格式化输出：
<Response><问题>请问中国的四大发明是什么？</问题><回复>中国的四大发明是造纸术、印刷术、火药和指南针，它们在各自的领域内发挥了重要作用，对世界历史的发展产生了深远的影响</回复></Response>


# 二，交互式推理

In [36]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_id = "/Users/wjx/Documents/wjx/minimind/minimind-3"
tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype=torch.bfloat16,
    device_map='auto'
)

system_message = {
    "role": "system",
    "content": "你是一个中文老师，必须用中文回答我的问题；现在你正在和学生讨论历史问题，这是个多轮对话场景，你需要结合历史谈话回答"
}

# 全局对话历史，后续对话一直往这里增加
message = [system_message]


def generate_response(user_input):
    # 用户输入加入历史
    message.append({"role": "user", "content": user_input})
    input_ids = tokenizer.apply_chat_template(message, add_generation_prompt=True, return_tensors="pt").to(model.device)

    # 终止标记
    terminators = [tokenizer.eos_token_id]
    outputs = model.generate(input_ids,
                             max_new_tokens=256,
                             eos_token_id=terminators,
                             do_sample=True,
                             temperature=0.6,
                             top_p=0.9, )
    # 解码
    output_text = tokenizer.decode(outputs[0][input_ids.shape[-1]:], skip_special_tokens=True)
    # 将模型回复增加到历史上下文中
    message.append({"role": "assistant", "content": output_text})
    return output_text


print("中文老师：同学！你可以问我任何问题，或者输入“退出”下课")

while True:
    user_input = input("You:")
    if user_input.lower() == "退出":
        print("下课")
        break
    else:
        reply = generate_response(user_input)
        print(f"回复：{reply}")

中文老师：同学！你可以问我任何问题，或者输入“退出”下课


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


回复：很高兴听到你最近在讨论历史问题。历史是一个复杂而多维的话题，涉及许多不同的方面。以下是一些可能的讨论点：

1. **古代文明与现代历史**：古代文明如古埃及、古希腊、古罗马等对世界历史有着深远的影响。这些文明在不同的时期和地区有着不同的发展和演变。例如，古埃及的金字塔和古罗马的金字塔在当时是世界上最高的建筑，而古希腊的宙斯则在公元前几世纪得到了广泛应用。

2. **科技与文明**：现代科技的发展对历史产生了深远的影响。例如，互联网的普及使得知识的传播和交流变得更加便捷。同时，工业革命也促进了知识的快速更新，如蒸汽机、电力、互联网等技术的快速发展。

3. **社会结构与政治制度**：历史事件往往对社会结构和政治制度产生深远影响。例如，二战后的世界格局发生了变化，引发了国际关系的复杂性。同时，历史事件的改变也影响了国家的政治制度，如中�


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


回复：刘备是一位中国历史上的著名政治家和军事家，他在政治、军事、文化和政治方面有着卓越的贡献。刘备在政治、军事、文化和政治方面都有着深刻的见解和贡献。

他曾经在公元前1世纪的三十年间担任重要职务，但最终在公元前1世纪的十年间，他因政治和军事权力而被视为“中国第一帝国”。刘备的军事才能和政治智慧在当时被视为“中国第一帝国”，他在历史上被认为是“中国第一帝国”。

刘备在政治上也有着卓越的贡献。他曾经在公元前1世纪的三十年间担任过重要职务，如公元前1世纪的十年间，他在公元前1世纪的十年间担任了重要职务，如公元前1世纪的十年间，他在公元前1世纪的十年间担任了重要职务，如公元前1世纪的十年间，他在公


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


回复：诸葛亮是中国历史上的著名政治家和军事家，他在政治、军事、文化和政治方面都有着深远的影响。诸葛亮在政治、军事、文化和政治方面都有着卓越的贡献。

他曾经担任过重要职务，如公元前1世纪的十年间，他在公元前1世纪的十年间担任过重要职务，如公元前1世纪的十年间，他在公元前1世纪的十年间担任了重要职务，如公元前1世纪的十年间，他在公元前1世纪的十年间担任了重要职务，如公元前1世纪的十年间，他在公元前1世纪的十年间担任了重要职务，如公元前1世纪的十年间，他在公元前1世纪的十年间担任了重要职务，如公元前1世纪的十年间，他在公元前1世纪的十年间担任了重要职务，如公元前1世纪的十年间，他在公


KeyboardInterrupt: Interrupted by user

# 三，基于Transformers的推理

In [42]:
import warnings
warnings.filterwarnings('ignore')
import torch
import torch.nn as nn
from torch import optim
import transformers
import numpy as np
import random

seed=42
print(f"MPS available: {torch.backends.mps.is_available()}")
print(f"MPS built: {torch.backends.mps.is_built()}")

MPS available: True
MPS built: True


In [50]:
import transformers
import torch
from IPython.display import display,HTML

# 加载模型
model_id = "/Users/wjx/Documents/wjx/minimind/minimind-3"
pipeline = transformers.pipeline(
    "text-generation",
    model=model_id,
    tokenizer=model_id,
    model_kwargs={"dtype": torch.bfloat16},
    device_map="auto",
)

Device set to use mps


In [58]:
# 构建提示模版
message = [
    {"role": "system", "content": "你是一个历史老师，当学生询问你问题的时候，你总是中文进行回答。"},
    {"role": "user", "content": "请问中国的四大发明是什么？"}
]
prompt = pipeline.tokenizer.apply_chat_template(
    message,
    tokenize=False,
    add_generation_prompt=True
)
terminators = [pipeline.tokenizer.eos_token_id]
outputs = pipeline(
    prompt,
    max_new_tokens=256,
    eos_token_id=terminators,
    do_sample=True, # 候选词随机选取
    temperature=0.6,
    top_p=0.9
)
print(outputs,'\n')
output_text = outputs[0]["generated_text"][len(prompt):]
print(f"问题: {message[1]['content']}\n回复: {output_text}")

[{'generated_text': '<|im_start|>system\n你是一个历史老师，当学生询问你问题的时候，你总是中文进行回答。<|im_end|>\n<|im_start|>user\n请问中国的四大发明是什么？<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n中国的四大发明是指造纸术、印刷术、火药和指南针。这些发明对世界历史产生了深远的影响，对人类文明的发展产生了重大影响。'}] 

问题: 请问中国的四大发明是什么？
回复: 中国的四大发明是指造纸术、印刷术、火药和指南针。这些发明对世界历史产生了深远的影响，对人类文明的发展产生了重大影响。


# 四，基于 Flask + transformers 框架，构建本地大模型问答系统

In [6]:
import transformers
import torch
from IPython.display import display, HTML
from flask import Flask, request, jsonify
import threading

app = Flask(__name__)
# 不转义中文
app.json.ensure_ascii = False
# 加载模型
model_id = "/Users/wjx/Documents/wjx/minimind/minimind-3"
pipeline = transformers.pipeline(
    "text-generation",
    model=model_id,
    tokenizer=model_id,
    model_kwargs={"dtype": torch.bfloat16},
    device_map="auto",
)

# 构建提示模版
message = [
    {"role": "system", "content": "你是一个历史老师，当学生询问你问题的时候，你总是中文进行回答。"},
    {"role": "user", "content": ""}
]

@app.route('/')
def home():
    return "Hello,Flask! This is a local LLM inference server."

@app.route('/predict', methods=['POST'])
def predict():
    data = request.get_json(force=True)
    if data is None:
        return jsonify({"error": "Invalid input JSON data"}), 400
    user_input = data.get('user_input', '')
    message[1]['content'] = user_input
    prompt = pipeline.tokenizer.apply_chat_template(
        message,
        tokenize=False,
        add_generation_prompt=True
    )
    terminators = [pipeline.tokenizer.eos_token_id]
    outputs = pipeline(
        prompt,
        max_new_tokens=256,
        eos_token_id=terminators,
        do_sample=True,  # 候选词随机选取
        temperature=0.6,
        top_p=0.9
    )
    output_text = outputs[0]["generated_text"][len(prompt):]
    return jsonify({"result": output_text})

PORT = 5002
threading.Thread(
    target=lambda: app.run(host="127.0.0.1", port=PORT, use_reloader=False),
    daemon=True
).start()
print(f"Flask 服务已在后台线程启动: http://127.0.0.1:{PORT}")

Device set to use mps


Flask 服务已在后台线程启动: http://127.0.0.1:5002
 * Serving Flask app '__main__'
 * Debug mode: off


Address already in use
Port 5002 is in use by another program. Either identify and stop that program, or start the server with a different port.


In [10]:
! curl http://127.0.0.1:5002

127.0.0.1 - - [26/Sep/2026 22:41:56] "GET / HTTP/1.1" 200 -


Hello,Flask! This is a local LLM inference server.

In [9]:
%%bash
curl -X POST http://127.0.0.1:5002/predict \
  -H "Content-Type: application/json" \
  -d '{"user_input": "请问中国的四大发明是什么？"}'

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0127.0.0.1 - - [26/Sep/2026 22:41:15] "POST /predict HTTP/1.1" 200 -
100   443  100   386  100    57   1063    157 --:--:-- --:--:-- --:--:--  1223


{"result":"\u4e2d\u56fd\u7684\u56db\u5927\u53d1\u660e\u6307\u7684\u662f\u9020\u7eb8\u672f\u3001\u5370\u5237\u672f\u3001\u706b\u836f\u548c\u6307\u5357\u9488\u3002\u8fd9\u56db\u9879\u53d1\u660e\u5bf9\u4e16\u754c\u5386\u53f2\u4ea7\u751f\u4e86\u6df1\u8fdc\u7684\u5f71\u54cd\uff0c\u5bf9\u4eba\u7c7b\u6587\u660e\u7684\u53d1\u5c55\u4ea7\u751f\u4e86\u5de8\u5927\u63a8\u52a8\u4f5c\u7528\u3002"}


In [11]:
import requests

resp = requests.post(
    "http://127.0.0.1:5002/predict",
    json={"user_input": "请问中国的四大发明是什么？"},
    proxies={"http": None, "https": None},
    timeout=300
)

print(resp.status_code)
print(resp.json())

127.0.0.1 - - [26/Sep/2026 22:42:03] "POST /predict HTTP/1.1" 200 -


200
{'result': '中国的四大发明分别是造纸术、印刷术、火药和指南针，它们分别对世界文明进程产生了深远的影响。\n\n1. **造纸术**：造纸术的发明者是蔡伦。蔡伦的蔡伦改进了造纸工艺，使得纸张的生产更加高效，便于书写和记录，同时也降低了纸张的生产成本。蔡伦的造纸术对后来的纸张生产产生了重大影响。\n\n2. **印刷术**：印刷术的发明者是毕达哥拉斯。毕达哥拉斯是印刷术的发明者，他使用印刷技术将书籍和其他文字都印在纸上，然后通过雕刻、打磨、涂覆等工艺，将文字或其他艺术形式印在纸上，然后通过雕刻、绘画等艺术形式进行印刷。印刷术的传播速度极快，几乎每小时能完成。\n\n3. **火药**：火药的发明者是瓦特。瓦特'}


### 5.停止服务

In [ ]:
# 停止 Flask 服务并释放端口（无需重启 kernel）
server.shutdown()
print("Flask 服务已停止，端口已释放")